# Tugas Sistem Temu Balik Informasi

Membandingkan performa dari metode ranked retrieval pada dataset News.csv. Konten yang diambil hanya pada kolom `content`.

Metode:
1. TF IDF (Cosine Similarity) (Vector Space)
    a. TF
    b. TF IDF Word2Vec
2. Query-Likelihood Retrieval Model (Probabilistic Approach)
    a. No Smoothing
    b. Add-One (Laplace)
    c. Linear Interpolation

In [ ]:
# Menghubungkan ke Google Drive (jika menggunakan Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    path_to_data = '/content/drive/MyDrive/STBI_Tugas1/News.csv' # Sesuaikan path ini jika Anda menyimpan di folder lain
except:
    print("Not running in Google Colab. Using local path.")
    path_to_data = 'News.csv'

Mounted at /content/drive


In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 38.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string
from collections import Counter
import math
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

## 1. Load & Preprocessing Data

In [ ]:
# Load dataset
try:
    df = pd.read_csv(path_to_data)
    # Gunakan hanya kolom 'content'
    if 'content' not in df.columns:
        print("Kolom 'content' tidak ditemukan. Pastikan dataset benar.")
    else:
        # Bersihkan missing values
        documents = df['content'].fillna('').tolist()
        print(f"Total documents: {len(documents)}")

        # UNTUK PERCOBAAN CEPAT (Bisa dihapus jika ingin menggunakan semua data)
        # Karena dataset bisa sangat besar, kita gunakan 5000 dokumen pertama untuk percobaan
        # documents = documents[:5000]
except Exception as e:
    print(f"Gagal memuat dataset: {e}")
    documents = []

Total documents: 14343


In [ ]:
stop_words = set(stopwords.words('indonesian'))

def preprocess(text):
    # Lowercase
    text = text.lower()
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stopwords
    tokens = [w for w in tokens if w not in stop_words]
    return tokens

print("Preprocessing documents...")
if len(documents) > 0:
    processed_docs = [preprocess(doc) for doc in documents]
    print("Preprocessing selesai.")

Preprocessing documents...
Preprocessing selesai.


## 2. Model 1: Vector Space Model

### a. TF (Cosine Similarity)

In [ ]:
# a. TF (menggunakan CountVectorizer dan Cosine Similarity)
if len(documents) > 0:
    vectorizer_tf = CountVectorizer(tokenizer=lambda x: x, preprocessor=lambda x: x, token_pattern=None)
    X_tf = vectorizer_tf.fit_transform(processed_docs)

    def search_tf(query, top_n=5):
        q_processed = preprocess(query)
        q_vec = vectorizer_tf.transform([q_processed])
        similarities = cosine_similarity(q_vec, X_tf).flatten()
        top_indices = similarities.argsort()[-top_n:][::-1]
        return [(i, similarities[i]) for i in top_indices if similarities[i] > 0]

### b. TF-IDF + Word2Vec

In [ ]:
if len(documents) > 0:
    # Latih Word2Vec dari dokumen kita sendiri agar vocabulary sesuai
    print("Melatih model Word2Vec...")
    w2v_model = Word2Vec(sentences=processed_docs, vector_size=100, window=5, min_count=1, workers=4)

    print("Menghitung TF-IDF weights...")
    vectorizer_tfidf = TfidfVectorizer(tokenizer=lambda x: x, preprocessor=lambda x: x, token_pattern=None)
    X_tfidf = vectorizer_tfidf.fit_transform(processed_docs)
    tfidf_features = vectorizer_tfidf.get_feature_names_out()
    tfidf_weights = dict(zip(tfidf_features, vectorizer_tfidf.idf_))

    def get_doc_vector(doc_tokens):
        vec = np.zeros(100)
        weight_sum = 0
        for token in doc_tokens:
            if token in w2v_model.wv and token in tfidf_weights:
                tf = doc_tokens.count(token) # Term frequency sederhana
                idf = tfidf_weights[token]
                tf_idf = tf * idf
                vec += w2v_model.wv[token] * tf_idf
                weight_sum += tf_idf
        if weight_sum > 0:
            vec /= weight_sum
        return vec

    print("Membangun representasi vektor dokumen dengan TF-IDF + Word2Vec...")
    doc_vectors = np.array([get_doc_vector(doc) for doc in processed_docs])

    def search_tfidf_w2v(query, top_n=5):
        q_processed = preprocess(query)
        q_vec = get_doc_vector(q_processed).reshape(1, -1)
        if np.sum(q_vec) == 0:
            return []
        similarities = cosine_similarity(q_vec, doc_vectors).flatten()
        top_indices = similarities.argsort()[-top_n:][::-1]
        return [(i, similarities[i]) for i in top_indices if similarities[i] > 0]

Melatih model Word2Vec...
Menghitung TF-IDF weights...
Membangun representasi vektor dokumen dengan TF-IDF + Word2Vec...


## 3. Model 2: Query-Likelihood Retrieval Model (Probabilistic Approach)

In [ ]:
if len(documents) > 0:
    # Bangun index dan statistik koleksi untuk model probabilitas
    print("Membangun statistik koleksi untuk QL Model...")
    doc_lengths = [len(doc) for doc in processed_docs]
    collection_length = sum(doc_lengths)

    collection_counts = Counter()
    for doc in processed_docs:
        collection_counts.update(doc)

    vocab_size = len(collection_counts)

    # Term frequencies per document (untuk akses cepat)
    doc_tfs = [Counter(doc) for doc in processed_docs]

Membangun statistik koleksi untuk QL Model...


### a. No Smoothing

In [ ]:
if len(documents) > 0:
    def search_ql_no_smoothing(query, top_n=5):
        q_processed = preprocess(query)
        scores = []
        for i, doc_tf in enumerate(doc_tfs):
            if doc_lengths[i] == 0:
                scores.append((i, float('-inf')))
                continue
            score = 0
            valid = True
            for q_term in q_processed:
                tf = doc_tf.get(q_term, 0)
                if tf == 0:
                    # Jika term tidak ada di dokumen, probabilitas jadi 0 (log(0) = -inf)
                    valid = False
                    break
                prob = tf / doc_lengths[i]
                score += math.log(prob)
            if valid:
                scores.append((i, score))
            else:
                scores.append((i, float('-inf')))

        # Sort berdasarkan score tertinggi
        scores = sorted(scores, key=lambda x: x[1], reverse=True)
        return [s for s in scores[:top_n] if s[1] != float('-inf')]

### b. Add-One (Laplace) Smoothing

In [ ]:
if len(documents) > 0:
    def search_ql_laplace(query, top_n=5):
        q_processed = preprocess(query)
        scores = []
        for i, doc_tf in enumerate(doc_tfs):
            score = 0
            doc_len = doc_lengths[i]
            for q_term in q_processed:
                tf = doc_tf.get(q_term, 0)
                # Probabilitas dengan smoothing Laplace
                prob = (tf + 1) / (doc_len + vocab_size)
                score += math.log(prob)
            scores.append((i, score))

        scores = sorted(scores, key=lambda x: x[1], reverse=True)
        return scores[:top_n]

### c. Linear Interpolation (Jelinek-Mercer)

In [ ]:
if len(documents) > 0:
    def search_ql_jelinek_mercer(query, lambda_param=0.5, top_n=5):
        q_processed = preprocess(query)
        scores = []
        for i, doc_tf in enumerate(doc_tfs):
            score = 0
            doc_len = doc_lengths[i]
            for q_term in q_processed:
                tf = doc_tf.get(q_term, 0)
                # P(w|d)
                p_ml_doc = tf / doc_len if doc_len > 0 else 0
                # P(w|C)
                p_ml_coll = collection_counts.get(q_term, 0) / collection_length if collection_length > 0 else 0

                # Linear Interpolation
                prob = (lambda_param * p_ml_doc) + ((1 - lambda_param) * p_ml_coll)

                if prob > 0:
                    score += math.log(prob)
                else:
                    # Jika term tidak ada di doc maupun di koleksi
                    score += float('-inf')
            scores.append((i, score))

        scores = sorted(scores, key=lambda x: x[1], reverse=True)
        return [s for s in scores[:top_n] if s[1] != float('-inf')]

## 4. Testing dan Perbandingan

In [ ]:
if len(documents) > 0:
    # 5 contoh query yang akan diuji
    queries = [
        "ppkm darurat warga negara asing dilarang masuk",
        "olimpiade tokyo bulu tangkis indonesia medali",
        "covid 19 isolasi mandiri protokol kesehatan",
        "bantuan sembako warga terdampak pandemi",
        "polisi korban tewas pembunuhan"
    ]

    for query_test in queries:
        print("================================================================================")
        print(f"QUERY: '{query_test}'")
        print("================================================================================\n")

        print("1.a. TF (Cosine Similarity)")
        res_tf = search_tf(query_test, top_n=10)
        for rank, (doc_id, score) in enumerate(res_tf):
            print(f"{rank+1}. Doc ID: {doc_id} | Score: {score:.4f} | Content: {documents[doc_id][:150]}...")

        print("\n1.b. TF-IDF + Word2Vec (Cosine Similarity)")
        res_w2v = search_tfidf_w2v(query_test, top_n=10)
        for rank, (doc_id, score) in enumerate(res_w2v):
            print(f"{rank+1}. Doc ID: {doc_id} | Score: {score:.4f} | Content: {documents[doc_id][:150]}...")

        print("\n2.a. Query-Likelihood (No Smoothing)")
        res_ql_ns = search_ql_no_smoothing(query_test, top_n=10)
        for rank, (doc_id, score) in enumerate(res_ql_ns):
            print(f"{rank+1}. Doc ID: {doc_id} | Score: {score:.4f} | Content: {documents[doc_id][:150]}...")

        print("\n2.b. Query-Likelihood (Add-One / Laplace)")
        res_ql_laplace = search_ql_laplace(query_test, top_n=10)
        for rank, (doc_id, score) in enumerate(res_ql_laplace):
            print(f"{rank+1}. Doc ID: {doc_id} | Score: {score:.4f} | Content: {documents[doc_id][:150]}...")

        print("\n2.c. Query-Likelihood (Linear Interpolation, lambda=0.5)")
        res_ql_jm = search_ql_jelinek_mercer(query_test, lambda_param=0.5, top_n=10)
        for rank, (doc_id, score) in enumerate(res_ql_jm):
            print(f"{rank+1}. Doc ID: {doc_id} | Score: {score:.4f} | Content: {documents[doc_id][:150]}...")
        print("\n\n")

QUERY: 'ppkm darurat warga negara asing dilarang masuk'

1.a. TF (Cosine Similarity)
1. Doc ID: 4350 | Score: 0.4602 | Content:  Masuknya Tenaga Kerja Asing (TKA) ke Indonesia terus menjadi sorotan masyarakat. Sebut saja ketika 20 orang pekerja asing asal China dikabarkan masuk...
2. Doc ID: 11203 | Score: 0.4200 | Content:   Kepala Kantor Wilayah Kementerian Hukum dan HAM (Kemenkumham) Bali Jamaruli Manihuruk menegaskan pihaknya akan langsung mendeportasi warga negara as...
3. Doc ID: 11316 | Score: 0.4171 | Content:   Selama Pandemi Covid-19, ada sebanyak 198 Warga Negara Asing (WNA) di Pulau Bali, dari tahun 2020 hingga 2021 yang dideportasi oleh Kantor Wilayah K...
4. Doc ID: 706 | Score: 0.4095 | Content:  PPPKM darurat akan kembali diperpanjang pemerintah hingga akhir Juli 2021. Ini diungkapkan Menteri Koordinator Bidang Pembangunan Manusia dan Kebuday...
5. Doc ID: 5229 | Score: 0.4088 | Content:   Pemerintah belum menutup pintu perjalanan internasional di tengah penyelenggaraan